# Курсовая — Полный пайплайн анализа видео

Пайплайн принимает **ссылку на видео** (YouTube / прямой mp4 / другие источники, поддерживаемые `yt-dlp`) и выполняет анализ:

1. Скачивание видео
2. Нарезка на кадры (ПЗ 2)
3. OCR субтитров — оптимизировано через перцептивный хеш (ПЗ 3)
4. Транскрипция речи Whisper (ПЗ 4)
5. Детекция объектов YOLOv8 (ПЗ 5)
6. Классификация кадров ResNet50 (ПЗ 6)
7. Описание сцен через LLM (ПЗ 7)
8. Постобработка и итоговый отчёт (ПЗ 8)

**Результаты сохраняются в** `/MyDrive/cv-frames/результаты/<имя_видео>/`.

## 1. Установка зависимостей

In [ ]:
!pip install yt-dlp easyocr ultralytics openai-whisper moviepy rapidfuzz torch torchvision tqdm requests pandas matplotlib -q
!apt-get install -y ffmpeg -q

## 2. Конфиг — вставьте ссылку на видео

In [ ]:
# === ВСТАВЬТЕ ССЫЛКУ НА ВИДЕО СЮДА ===
VIDEO_URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"

# === Параметры обработки ===
FRAME_STEP        = 30      # каждый N-й кадр для анализа (30 = ~1 в секунду при 30fps)
CONF_YOLO         = 0.4     # порог уверенности YOLO
OCR_DEDUP_THR     = 85      # порог fuzzy-дедупликации OCR (0-100)
DET_MERGE_WINDOW  = 5       # склейка YOLO-событий в пределах N кадров
LLM_INTERVAL_SEC  = 5       # интервал LLM-описаний (раз в N секунд видео)
WHISPER_MODEL     = "base"  # base / small / medium

# === Адрес LLM сервера (ПЗ 7) ===
LLM_API_URL = "http://92.51.37.40:8000/describe"

import os, json, shutil, torch
from google.colab import drive, files

drive.mount('/content/drive', force_remount=True)

BASE_DRIVE = '/content/drive/MyDrive/cv-frames'
os.makedirs(BASE_DRIVE, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'устройство: {DEVICE}')

## 3. Скачивание видео по ссылке

In [ ]:
import subprocess, re

WORK_DIR = '/content/work'
os.makedirs(WORK_DIR, exist_ok=True)
VIDEO_PATH = f'{WORK_DIR}/video.mp4'

if os.path.exists(VIDEO_PATH):
    os.remove(VIDEO_PATH)

print(f'скачиваем: {VIDEO_URL}')
result = subprocess.run(
    ['yt-dlp', '-f', 'mp4/best[ext=mp4]/best', '-o', VIDEO_PATH, VIDEO_URL],
    capture_output=True, text=True, timeout=600
)
if result.returncode != 0:
    raise RuntimeError(f'yt-dlp ошибка: {result.stderr}')

# имя видео для папки результатов
VIDEO_NAME = re.sub(r'[^\w\-]', '_', VIDEO_URL.split('/')[-1])[:50] or 'video'
RESULTS_DIR = f'{BASE_DRIVE}/результаты/{VIDEO_NAME}'
FRAMES_DIR  = f'{WORK_DIR}/frames'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)

size_mb = os.path.getsize(VIDEO_PATH) / 1024 / 1024
print(f'видео скачано: {size_mb:.1f} МБ')
print(f'результаты будут здесь: {RESULTS_DIR}')

## 4. Нарезка видео на кадры (ПЗ 2)

In [ ]:
import cv2
from tqdm.notebook import tqdm

cap   = cv2.VideoCapture(VIDEO_PATH)
fps   = cap.get(cv2.CAP_PROP_FPS) or 25
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration_sec = total / fps

frame_idx = saved = 0
for _ in tqdm(range(total), desc='нарезка кадров'):
    ret, frame = cap.read()
    if not ret:
        break
    if frame_idx % FRAME_STEP == 0:
        cv2.imwrite(f'{FRAMES_DIR}/frame_{frame_idx:06d}.jpg', frame)
        saved += 1
    frame_idx += 1
cap.release()

FRAME_LIST = sorted(os.listdir(FRAMES_DIR))
print(f'длительность: {duration_sec:.1f} с | fps: {fps:.1f} | сохранено кадров: {saved}')

## 5. OCR субтитров — оптимизированный (ПЗ 3)

Главная оптимизация: **перцептивный хеш (dHash)** субтитровой зоны. Если на текущем кадре субтитр визуально совпадает с предыдущим обработанным — пропускаем дорогой OCR. Это даёт ускорение в 4–8 раз без потери точности, так как одни и те же субтитры обычно держатся 2–4 секунды.

Дополнительно:
- Обрабатываем только нижние 20% кадра (зона субтитров)
- Resize crop'а до 60% — текст всё ещё читается, OCR работает быстрее
- Порог уверенности 0.4 отсекает мусор

In [ ]:
import easyocr
import numpy as np
import pandas as pd

# === Инициализация EasyOCR ===
# Здесь мы загружаем модель распознавания текста с поддержкой русского и английского.
# Параметр gpu=True ускоряет работу в 5-10 раз по сравнению с CPU.
reader = easyocr.Reader(['ru', 'en'], gpu=(DEVICE == 'cuda'))

SUBTITLE_FRACTION = 0.20  # доля кадра снизу, в которой ищем субтитры


# === Функция dhash ===
# Здесь мы считаем перцептивный хеш изображения: уменьшаем картинку до 9x8,
# сравниваем соседние пиксели по строкам и получаем 64-битный отпечаток.
# Если на двух кадрах субтитр визуально один и тот же - хеши совпадут,
# и мы сможем пропустить дорогой OCR на повторе.
def dhash(img, size=8):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
    r = cv2.resize(g, (size + 1, size))
    return (r[:, 1:] > r[:, :-1]).tobytes()


# === Функция extract_subtitle_crop ===
# Здесь мы вырезаем нижнюю часть кадра (там где субтитры)
# и уменьшаем её до 60% размера для ускорения OCR без потери читаемости текста.
def extract_subtitle_crop(img):
    h = img.shape[0]
    crop = img[int(h * (1 - SUBTITLE_FRACTION)):, :]
    small = cv2.resize(crop, None, fx=0.6, fy=0.6, interpolation=cv2.INTER_AREA)
    return crop, small


# === Функция run_ocr_on_frame ===
# Здесь мы запускаем EasyOCR на подготовленном crop'е и собираем
# распознанный текст с уверенностью выше 0.4 (отсекаем шум).
def run_ocr_on_frame(crop_small):
    gray = cv2.cvtColor(crop_small, cv2.COLOR_BGR2GRAY)
    results = []
    for _, text, prob in reader.readtext(gray):
        text = text.strip()
        if text and prob > 0.4:
            results.append((text, round(prob, 3)))
    return results


# === Основной цикл OCR ===
# Здесь мы проходим по всем кадрам, считаем хеш субтитровой зоны и
# пропускаем кадр если субтитр не изменился с прошлого раза.
# На реально различающихся кадрах вызываем OCR.
ocr_rows = []
prev_hash = None
skipped = 0

for fname in tqdm(FRAME_LIST, desc='OCR'):
    img = cv2.imread(f'{FRAMES_DIR}/{fname}')
    if img is None:
        continue

    crop, small = extract_subtitle_crop(img)
    h_now = dhash(crop)
    if h_now == prev_hash:
        skipped += 1
        continue
    prev_hash = h_now

    for text, conf in run_ocr_on_frame(small):
        ocr_rows.append({'frame': fname, 'text': text, 'conf': conf})

df_ocr = pd.DataFrame(ocr_rows)
df_ocr.to_csv(f'{RESULTS_DIR}/ocr.csv', index=False, encoding='utf-8-sig')

print(f'кадров пропущено по dHash: {skipped} из {len(FRAME_LIST)}')
print(f'распознано строк: {len(df_ocr)}')
df_ocr.head(20)

## 6. Транскрипция речи Whisper (ПЗ 4)

In [ ]:
import whisper
from moviepy.editor import VideoFileClip

audio_path = f'{WORK_DIR}/audio.wav'
VideoFileClip(VIDEO_PATH).audio.write_audiofile(audio_path, verbose=False, logger=None)

w_model = whisper.load_model(WHISPER_MODEL, device=DEVICE)
w_result = w_model.transcribe(audio_path)

transcript = w_result['text']
segments   = w_result.get('segments', [])

with open(f'{RESULTS_DIR}/transcript.txt', 'w', encoding='utf-8') as f:
    f.write(transcript)

print(f'язык: {w_result.get("language", "?")}')
print(f'сегментов: {len(segments)}')
print(transcript[:500])

## 7. Детекция объектов YOLOv8 (ПЗ 5)

In [ ]:
from ultralytics import YOLO

yolo = YOLO('yolov8n.pt')
yolo_rows = []

for fname in tqdm(FRAME_LIST, desc='YOLO'):
    for r in yolo(f'{FRAMES_DIR}/{fname}', conf=CONF_YOLO, verbose=False):
        for b in r.boxes:
            cls_id = int(b.cls[0])
            yolo_rows.append({
                'frame_num': int(fname.split('_')[1].split('.')[0]),
                'frame': fname,
                'class': r.names[cls_id],
                'conf': round(float(b.conf[0]), 3),
                'x1': int(b.xyxy[0][0]), 'y1': int(b.xyxy[0][1]),
                'x2': int(b.xyxy[0][2]), 'y2': int(b.xyxy[0][3]),
            })

df_yolo = pd.DataFrame(yolo_rows)
df_yolo.to_csv(f'{RESULTS_DIR}/yolo_detections.csv', index=False)
print(f'детекций: {len(df_yolo)}')
print(df_yolo['class'].value_counts().head(10))

## 8. Классификация кадров ResNet50 (ПЗ 6)

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms
from PIL import Image

weights  = ResNet50_Weights.DEFAULT
resnet   = resnet50(weights=weights).to(DEVICE).eval()
preprocess = weights.transforms()
class_names = weights.meta['categories']

cls_rows = []
BATCH = 16
batch_imgs, batch_names = [], []

def flush():
    if not batch_imgs: return
    x = torch.stack(batch_imgs).to(DEVICE)
    with torch.no_grad():
        out = resnet(x).softmax(1)
    top5 = out.topk(5, dim=1)
    for i, name in enumerate(batch_names):
        cls_rows.append({
            'frame': name,
            'top5_classes': [class_names[j] for j in top5.indices[i].tolist()],
            'top5_probs':   [round(p, 3) for p in top5.values[i].tolist()],
        })
    batch_imgs.clear(); batch_names.clear()

for fname in tqdm(FRAME_LIST, desc='ResNet'):
    img = Image.open(f'{FRAMES_DIR}/{fname}').convert('RGB')
    batch_imgs.append(preprocess(img))
    batch_names.append(fname)
    if len(batch_imgs) >= BATCH:
        flush()
flush()

df_cls = pd.DataFrame(cls_rows)
df_cls.to_csv(f'{RESULTS_DIR}/classifications.csv', index=False)
print(f'классифицировано: {len(df_cls)}')

## 9. Описание сцен через LLM (ПЗ 7)

In [ ]:
import requests, base64, time

# выбираем кадры с шагом LLM_INTERVAL_SEC секунд
step_frames = max(1, int(fps * LLM_INTERVAL_SEC / FRAME_STEP))
llm_targets = FRAME_LIST[::step_frames]

llm_results = []
for fname in tqdm(llm_targets, desc='LLM'):
    try:
        with open(f'{FRAMES_DIR}/{fname}', 'rb') as f:
            r = requests.post(LLM_API_URL, files={'file': f}, timeout=120)
        r.raise_for_status()
        objects = r.json().get('objects', '')
        frame_num = int(fname.split('_')[1].split('.')[0])
        llm_results.append({
            'frame': fname,
            'time_sec': round(frame_num / fps, 1),
            'objects': objects,
        })
    except Exception as e:
        llm_results.append({'frame': fname, 'error': str(e)})
    time.sleep(0.3)  # щадим rate-limit

with open(f'{RESULTS_DIR}/llm_descriptions.json', 'w', encoding='utf-8') as f:
    json.dump(llm_results, f, ensure_ascii=False, indent=2)

print(f'описаний LLM: {len(llm_results)}')
for r in llm_results[:5]:
    print(r)

## 10. Постобработка + итоговый отчёт (ПЗ 8)

In [ ]:
from rapidfuzz import fuzz, process
from collections import Counter
import matplotlib.pyplot as plt

# === OCR: дедупликация почти-одинаковых строк ===
unique_texts = []
for t in df_ocr['text'].dropna().tolist():
    if not unique_texts:
        unique_texts.append(t); continue
    best = process.extractOne(t, unique_texts, scorer=fuzz.ratio)
    if best is None or best[1] < OCR_DEDUP_THR:
        unique_texts.append(t)

# === YOLO: склейка детекций в события ===
yolo_events = []
if not df_yolo.empty:
    for cls, grp in df_yolo.groupby('class'):
        grp = grp.sort_values('frame_num').reset_index(drop=True)
        start = grp.iloc[0]['frame_num']; prev = start
        for fn in grp['frame_num'][1:]:
            if fn - prev > DET_MERGE_WINDOW * FRAME_STEP:
                yolo_events.append({'class': cls, 'start_frame': int(start), 'end_frame': int(prev)})
                start = fn
            prev = fn
        yolo_events.append({'class': cls, 'start_frame': int(start), 'end_frame': int(prev)})

df_events = pd.DataFrame(yolo_events)
if not df_events.empty:
    df_events.to_csv(f'{RESULTS_DIR}/yolo_events.csv', index=False)

# === LLM: подсчёт частых объектов ===
llm_objects = Counter()
for r in llm_results:
    for obj in (r.get('objects') or '').split(','):
        obj = obj.strip().lower()
        if obj: llm_objects[obj] += 1

# === Графики ===
charts_dir = f'{RESULTS_DIR}/charts'
os.makedirs(charts_dir, exist_ok=True)

if not df_yolo.empty:
    top10 = df_yolo['class'].value_counts().head(10)
    plt.figure(figsize=(10, 4))
    top10.plot(kind='bar'); plt.title('YOLO: топ-10 объектов'); plt.tight_layout()
    plt.savefig(f'{charts_dir}/yolo_top10.png'); plt.close()

if llm_objects:
    top10 = dict(llm_objects.most_common(10))
    plt.figure(figsize=(10, 4))
    plt.bar(top10.keys(), top10.values()); plt.xticks(rotation=45, ha='right')
    plt.title('LLM: топ-10 объектов'); plt.tight_layout()
    plt.savefig(f'{charts_dir}/llm_top10.png'); plt.close()

# === Итоговый отчёт ===
report = {
    'video_url':       VIDEO_URL,
    'duration_sec':    round(duration_sec, 1),
    'frames_analyzed': len(FRAME_LIST),
    'fps':             round(fps, 1),
    'ocr': {
        'total_lines':   int(len(df_ocr)),
        'unique_lines':  len(unique_texts),
        'sample':        unique_texts[:20],
    },
    'whisper': {
        'language':  w_result.get('language'),
        'segments':  len(segments),
        'transcript_preview': transcript[:500],
    },
    'yolo': {
        'detections':    int(len(df_yolo)),
        'events':        int(len(df_events)),
        'top_classes':   df_yolo['class'].value_counts().head(10).to_dict() if not df_yolo.empty else {},
    },
    'resnet': {
        'frames_classified': int(len(df_cls)),
    },
    'llm': {
        'frames_described': len(llm_results),
        'top_objects':      dict(llm_objects.most_common(10)),
    },
}

report_path = f'{RESULTS_DIR}/final_report.json'
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))
print(f'\nотчёт сохранён: {report_path}')

## 11. Скачать отчёт

In [ ]:
files.download(report_path)